# GPT-OSS: Inference

Load pre-trained weights and generate text using the Nano GPT-OSS model.

**Prerequisites:** Run `train.ipynb` first, or download published weights.

---

### Inference Pipeline
```
1. Define architecture (must match training exactly)
2. Load tokenizer (same Harmony BPE)
3. Load checkpoint → restore weights into model
4. Auto-regressive generation:
   prompt → tokenize → model(tokens) → logits → sample → append → repeat
```

### Sampling Strategies
- **Temperature**: Controls randomness. Low (0.3) = safe/repetitive, High (1.2) = creative/risky
- **Top-K**: Only consider the K most probable next tokens (ignore the rest)
- **Top-P (Nucleus)**: Keep smallest set of tokens whose cumulative probability exceeds P
- **Greedy (temp=0)**: Always pick the most probable token (deterministic but boring)

## 1. Setup

In [ ]:
!pip install -q torch tiktoken

In [ ]:
import os
import json
import math
import time
from dataclasses import dataclass

import torch
import torch.nn.functional as F
import torch.distributed as dist
import tiktoken

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Architecture (same as training)

The model architecture must be defined identically to training so `load_state_dict()` can map saved weights to the correct layers. This is a self-contained copy.

**Components (see train.ipynb for detailed comments):**
- `RMSNorm` — normalize without mean subtraction (faster than LayerNorm)
- `RotaryEmbedding` — encode position by rotating Q,K vectors (not additive)
- `sdpa()` — scaled dot-product attention with GQA + sliding window + sinks
- `AttentionBlock` — norm → QKV → RoPE → SDPA → project → residual
- `MLPBlock` — MoE: router → top-K experts (SwiGLU) → weighted sum → residual
- `GPToss` — embedding → 12 transformer blocks → norm → unembedding → logits

In [ ]:
@dataclass
class ModelConfig:
    """Must match the config used during training (loaded from checkpoint)."""
    num_hidden_layers: int = 12
    num_experts: int = 4
    experts_per_token: int = 2
    vocab_size: int = 201088
    hidden_size: int = 1024
    intermediate_size: int = 1024
    swiglu_limit: float = 7.0
    head_dim: int = 64
    num_attention_heads: int = 16
    num_key_value_heads: int = 4
    sliding_window: int = 128
    initial_context_length: int = 4096
    rope_theta: float = 150000.0
    rope_scaling_factor: float = 1.0
    rope_ntk_alpha: float = 1.0
    rope_ntk_beta: float = 32.0


class RMSNorm(torch.nn.Module):
    """x / RMS(x) * scale — no mean subtraction, ~15% faster than LayerNorm."""
    def __init__(self, num_features, eps=1e-05, device=None):
        super().__init__()
        self.num_features = num_features
        self.eps = eps
        self.scale = torch.nn.Parameter(
            torch.ones(num_features, device=device, dtype=torch.float32)
        )

    def forward(self, x):
        t = x.float()
        t = t * torch.rsqrt(torch.mean(t**2, dim=-1, keepdim=True) + self.eps)
        return (t * self.scale).to(x.dtype)


def _apply_rotary_emb(x, cos, sin):
    """Rotate pairs of dimensions: encodes position via 2D rotation."""
    cos = cos.unsqueeze(-2).to(x.dtype)
    sin = sin.unsqueeze(-2).to(x.dtype)
    x1, x2 = torch.chunk(x, 2, dim=-1)
    o1 = x1 * cos - x2 * sin
    o2 = x2 * cos + x1 * sin
    return torch.cat((o1, o2), dim=-1)


class RotaryEmbedding(torch.nn.Module):
    """RoPE with YaRN scaling for context extension beyond training length."""
    def __init__(self, head_dim, base, dtype, initial_context_length=4096,
                 scaling_factor=1.0, ntk_alpha=1.0, ntk_beta=32.0, device=None):
        super().__init__()
        self.head_dim = head_dim
        self.base = base
        self.dtype = dtype
        self.initial_context_length = initial_context_length
        self.scaling_factor = scaling_factor
        self.ntk_alpha = ntk_alpha
        self.ntk_beta = ntk_beta
        self.device = device

    def _compute_concentration_and_inv_freq(self):
        freq = self.base ** (
            torch.arange(0, self.head_dim, 2, dtype=torch.float, device=self.device)
            / self.head_dim
        )
        if self.scaling_factor > 1.0:
            concentration = 0.1 * math.log(self.scaling_factor) + 1.0
            d_half = self.head_dim / 2
            low = d_half * math.log(self.initial_context_length / (self.ntk_beta * 2 * math.pi)) / math.log(self.base)
            high = d_half * math.log(self.initial_context_length / (self.ntk_alpha * 2 * math.pi)) / math.log(self.base)
            interpolation = 1.0 / (self.scaling_factor * freq)
            extrapolation = 1.0 / freq
            ramp = (torch.arange(d_half, dtype=torch.float32, device=freq.device) - low) / (high - low)
            mask = 1 - ramp.clamp(0, 1)
            inv_freq = interpolation * (1 - mask) + extrapolation * mask
        else:
            concentration = 1.0
            inv_freq = 1.0 / freq
        return concentration, inv_freq

    def _compute_cos_sin(self, num_tokens):
        concentration, inv_freq = self._compute_concentration_and_inv_freq()
        t = torch.arange(num_tokens, dtype=torch.float32, device=self.device)
        freqs = torch.einsum("i,j->ij", t, inv_freq)
        return freqs.cos() * concentration, freqs.sin() * concentration

    def forward(self, query, key):
        num_tokens = query.shape[0]
        cos, sin = self._compute_cos_sin(num_tokens)
        query_shape = query.shape
        query = _apply_rotary_emb(query.view(num_tokens, -1, self.head_dim), cos, sin).reshape(query_shape)
        key_shape = key.shape
        key = _apply_rotary_emb(key.view(num_tokens, -1, self.head_dim), cos, sin).reshape(key_shape)
        return query, key


def sdpa(Q, K, V, S, sm_scale, sliding_window=0):
    """Scaled dot-product attention with GQA broadcast, sliding mask, and sinks."""
    n_tokens, n_heads, q_mult, d_head = Q.shape
    K = K[:, :, None, :].expand(-1, -1, q_mult, -1)
    V = V[:, :, None, :].expand(-1, -1, q_mult, -1)
    S = S.reshape(n_heads, q_mult, 1, 1).expand(-1, -1, n_tokens, -1)
    mask = torch.triu(Q.new_full((n_tokens, n_tokens), -float("inf")), diagonal=1)
    if sliding_window > 0:
        mask += torch.tril(mask.new_full((n_tokens, n_tokens), -float("inf")), diagonal=-sliding_window)
    QK = torch.einsum("qhmd,khmd->hmqk", Q, K) * sm_scale
    QK += mask[None, None, :, :]
    QK = torch.cat([QK, S], dim=-1)
    W = torch.softmax(QK, dim=-1)[..., :-1]
    return torch.einsum("hmqk,khmd->qhmd", W, V).reshape(n_tokens, -1)


class AttentionBlock(torch.nn.Module):
    """GQA attention with RoPE, alternating sliding/full causal, and sinks."""
    def __init__(self, config, layer_idx=0, device=None):
        super().__init__()
        self.head_dim = config.head_dim
        self.num_attention_heads = config.num_attention_heads
        self.num_key_value_heads = config.num_key_value_heads
        self.sliding_window = config.sliding_window if layer_idx % 2 == 0 else 0
        self.sinks = torch.nn.Parameter(torch.empty(config.num_attention_heads, device=device, dtype=torch.bfloat16))
        self.norm = RMSNorm(config.hidden_size, device=device)
        qkv_dim = config.head_dim * (config.num_attention_heads + 2 * config.num_key_value_heads)
        self.qkv = torch.nn.Linear(config.hidden_size, qkv_dim, device=device, dtype=torch.bfloat16)
        self.out = torch.nn.Linear(config.head_dim * config.num_attention_heads, config.hidden_size, device=device, dtype=torch.bfloat16)
        self.sm_scale = 1 / math.sqrt(config.head_dim)
        self.rope = RotaryEmbedding(config.head_dim, config.rope_theta, torch.float32,
                                    initial_context_length=config.initial_context_length,
                                    scaling_factor=config.rope_scaling_factor,
                                    ntk_alpha=config.rope_ntk_alpha, ntk_beta=config.rope_ntk_beta, device=device)

    def forward(self, x):
        t = self.norm(x)
        qkv = self.qkv(t)
        q = qkv[:, :self.num_attention_heads * self.head_dim].contiguous()
        k = qkv[:, self.num_attention_heads * self.head_dim:(self.num_attention_heads + self.num_key_value_heads) * self.head_dim].contiguous()
        v = qkv[:, (self.num_attention_heads + self.num_key_value_heads) * self.head_dim:(self.num_attention_heads + 2 * self.num_key_value_heads) * self.head_dim].contiguous()
        q = q.view(-1, self.num_key_value_heads, self.num_attention_heads // self.num_key_value_heads, self.head_dim)
        k = k.view(-1, self.num_key_value_heads, self.head_dim)
        v = v.view(-1, self.num_key_value_heads, self.head_dim)
        q, k = self.rope(q, k)
        t = sdpa(q, k, v, self.sinks, self.sm_scale, self.sliding_window)
        return x + self.out(t)


def swiglu(x, alpha=1.702, limit=7.0):
    """SwiGLU: Swish(gate) * (linear + 1) — gated activation for FFN."""
    x_glu, x_linear = x[..., ::2], x[..., 1::2]
    x_glu = x_glu.clamp(max=limit)
    x_linear = x_linear.clamp(-limit, limit)
    return (x_glu * torch.sigmoid(alpha * x_glu)) * (x_linear + 1)


class MLPBlock(torch.nn.Module):
    """Mixture of Experts: route each token to top-K experts, blend outputs."""
    def __init__(self, config, device=None):
        super().__init__()
        self.num_experts = config.num_experts
        self.experts_per_token = config.experts_per_token
        self.swiglu_limit = config.swiglu_limit
        self.world_size = dist.get_world_size() if dist.is_initialized() else 1
        self.norm = RMSNorm(config.hidden_size, device=device)
        self.gate = torch.nn.Linear(config.hidden_size, config.num_experts, device=device, dtype=torch.bfloat16)
        self.experts = torch.nn.ModuleList([
            torch.nn.Sequential(
                torch.nn.Linear(config.hidden_size, config.intermediate_size * 2 // self.world_size, device=device, dtype=torch.bfloat16),
                torch.nn.Linear(config.intermediate_size // self.world_size, config.hidden_size, device=device, dtype=torch.bfloat16)
            ) for _ in range(config.num_experts)
        ])

    def forward(self, x):
        seq_len, hidden_size = x.shape
        t = self.norm(x)
        g = self.gate(t)
        experts = torch.topk(g, k=self.experts_per_token, dim=-1, sorted=True)
        expert_weights = F.softmax(experts.values, dim=-1)
        expert_indices = experts.indices
        t_flat = t.view(-1, hidden_size)
        expert_indices_flat = expert_indices.view(-1, self.experts_per_token)
        expert_weights_flat = expert_weights.view(-1, self.experts_per_token)
        output = torch.zeros_like(t_flat)
        for expert_idx in range(self.num_experts):
            mask = (expert_indices_flat == expert_idx).any(dim=-1)
            if not mask.any():
                continue
            token_indices = torch.where(mask)[0]
            expert_pos = (expert_indices_flat[token_indices] == expert_idx).nonzero(as_tuple=True)[1]
            expert_input = t_flat[token_indices]
            weights = expert_weights_flat[token_indices, expert_pos]
            expert_out = swiglu(self.experts[expert_idx][0](expert_input), limit=self.swiglu_limit)
            expert_out = self.experts[expert_idx][1](expert_out)
            output[token_indices] += expert_out * weights.unsqueeze(-1)
        if self.world_size > 1:
            dist.all_reduce(output, op=dist.ReduceOp.SUM)
        return x + output.view(seq_len, hidden_size)


class TransformerBlock(torch.nn.Module):
    """Attention + MoE with residual connections (pre-norm architecture)."""
    def __init__(self, config, layer_idx, device=None):
        super().__init__()
        self.attn = AttentionBlock(config, layer_idx, device)
        self.mlp = MLPBlock(config, device)

    def forward(self, x):
        return self.mlp(self.attn(x))


class GPToss(torch.nn.Module):
    """Full model: embed → 12 blocks → norm → unembedding → vocab logits."""
    def __init__(self, config, device=None):
        super().__init__()
        self.config = config
        self.embedding = torch.nn.Embedding(config.vocab_size, config.hidden_size, device=device, dtype=torch.bfloat16)
        self.blocks = torch.nn.ModuleList([TransformerBlock(config, i, device) for i in range(config.num_hidden_layers)])
        self.norm = RMSNorm(config.hidden_size, device=device)
        self.unembedding = torch.nn.Linear(config.hidden_size, config.vocab_size, bias=False, device=device, dtype=torch.bfloat16)

    def forward(self, x):
        x = self.embedding(x)
        for block in self.blocks:
            x = block(x)
        return self.unembedding(self.norm(x))

## 3. Load Tokenizer

In [ ]:
def get_tokenizer():
    """Build the same Harmony tokenizer used during training.
    Must be identical — different tokenizer = garbage output."""
    o200k_base = tiktoken.get_encoding("o200k_base")
    return tiktoken.Encoding(
        name="o200k_harmony",
        pat_str=o200k_base._pat_str,
        mergeable_ranks=o200k_base._mergeable_ranks,
        special_tokens={
            **o200k_base._special_tokens,
            "<|startoftext|>": 199998, "<|endoftext|>": 199999,
            "<|reserved_200000|>": 200000, "<|reserved_200001|>": 200001,
            "<|return|>": 200002, "<|constrain|>": 200003,
            "<|reserved_200004|>": 200004, "<|channel|>": 200005,
            "<|start|>": 200006, "<|end|>": 200007,
            "<|message|>": 200008, "<|reserved_200009|>": 200009,
            "<|reserved_200010|>": 200010, "<|reserved_200011|>": 200011,
            "<|call|>": 200012,
        } | {f"<|reserved_{i}|>": i for i in range(200013, 201088)},
    )

tokenizer = get_tokenizer()
print(f"Tokenizer loaded — vocab size: {tokenizer.n_vocab}")

## 4. Load Weights

The checkpoint contains:
- `model_state_dict` — all learned parameters (weights and biases)
- `config` — architecture hyperparameters (ensures we build the right shaped model)
- `training_info` — metadata about the training run

In [ ]:
# Path to your saved checkpoint
CHECKPOINT_PATH = "checkpoints/gptoss_best.pt"  # or gptoss_final.pt

# Alternatively load from Google Drive:
# CHECKPOINT_PATH = "/content/drive/MyDrive/gptoss_weights/gptoss_best.pt"

print(f"Loading checkpoint: {CHECKPOINT_PATH}")
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)

# Reconstruct config from saved hyperparameters
# This ensures the model shape matches the saved weights exactly
config = ModelConfig(**checkpoint["config"])
print(f"Config loaded: {checkpoint['config']}")

if "training_info" in checkpoint:
    info = checkpoint["training_info"]
    print(f"\nTraining info:")
    print(f"  Best val loss: {info.get('best_val_loss', 'N/A')}")
    print(f"  Total steps: {info.get('total_steps', 'N/A')}")
    print(f"  Training time: {info.get('training_time_minutes', 'N/A'):.1f} min")

In [ ]:
# Build model with the SAME config, then load the learned weights
model = GPToss(config, device=DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()  # Disable dropout, set batch norm to eval mode (if any)

total_params = sum(p.numel() for p in model.parameters())
print(f"\nModel loaded: {total_params / 1e6:.1f}M parameters")
print(f"Ready for inference on {DEVICE}")

## 5. Text Generation

**Auto-regressive generation loop:**
```
tokens = tokenize(prompt)
repeat max_tokens times:
    logits = model(tokens[-context_len:])  # Forward pass
    next_logits = logits[-1]                # Only the LAST position predicts next token
    next_logits /= temperature              # Scale randomness
    filter by top_k / top_p                 # Remove unlikely options
    next_token = sample(softmax(logits))    # Pick one from the distribution
    tokens.append(next_token)               # Grow the sequence
```

Note: This is O(n^2) per token because we recompute attention for ALL tokens.
Production systems use KV-cache to make it O(n) per new token.

In [ ]:
CONTEXT_LEN = config.initial_context_length

@torch.inference_mode()  # Disables gradient tracking — faster + less memory
def generate(prompt, max_tokens=200, temperature=0.8, top_k=50, top_p=0.9):
    """Generate text auto-regressively from a prompt.

    Args:
        prompt: Input text string
        max_tokens: Maximum new tokens to generate
        temperature: >1 = more random, <1 = more focused, 0 = greedy
        top_k: Keep only K most probable tokens (0 = disabled)
        top_p: Nucleus sampling — keep smallest set summing to P
    
    Returns:
        Full generated text (prompt + continuation)
    """
    # Tokenize prompt: "Hello world" → [15496, 1917]
    idx = torch.tensor(tokenizer.encode(prompt), dtype=torch.int32).to(DEVICE)

    for _ in range(max_tokens):
        # Truncate to context window (model can only see this many tokens)
        idx_cond = idx[-CONTEXT_LEN:]
        
        # Forward pass: get logits for ALL positions, but we only need the last one
        # logits[-1, :] gives scores for what comes AFTER the last input token
        logits = model(idx_cond)[-1, :]  # (vocab_size,) — score per vocab token

        if temperature > 0:
            # TEMPERATURE SCALING:
            # Dividing by T < 1 makes distribution sharper (more confident)
            # Dividing by T > 1 makes distribution flatter (more exploratory)
            logits = logits / temperature

            # TOP-K FILTERING:
            # Zero out everything below the K-th highest score
            # This prevents sampling very unlikely tokens
            if top_k > 0:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[-1]] = -float('inf')  # -inf → 0 probability after softmax

            # TOP-P (NUCLEUS) FILTERING:
            # Keep the smallest set of tokens whose cumulative prob >= p
            # Adapts K dynamically: fewer tokens when model is confident
            if top_p < 1.0:
                sorted_logits, sorted_indices = torch.sort(logits, descending=True)
                cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
                # Remove tokens with cumulative probability above the threshold
                sorted_indices_to_remove = cumulative_probs > top_p
                # Shift right: keep at least the top-1 token
                sorted_indices_to_remove[1:] = sorted_indices_to_remove[:-1].clone()
                sorted_indices_to_remove[0] = False
                indices_to_remove = sorted_indices[sorted_indices_to_remove]
                logits[indices_to_remove] = -float('inf')

            # Convert logits → probabilities → sample one token
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)  # Weighted random choice
        else:
            # GREEDY DECODING: always pick the most probable token
            # Deterministic but can be repetitive/boring
            idx_next = torch.argmax(logits, dim=-1, keepdim=True)

        # Append new token and continue
        idx = torch.cat((idx, idx_next), dim=0)

    # Decode token IDs back to text: [15496, 1917, ...] → "Hello world ..."
    return tokenizer.decode(idx.tolist())

## 6. Interactive Generation

In [ ]:
prompts = [
    "Once upon a time, there was a little girl named",
    "The brave knight rode his horse through the",
    "A tiny bird sat on the tree and sang a",
    "Mom told Tim that he should always be",
    "One sunny morning, the cat went to the",
]

print("=" * 70)
print("GPT-OSS TEXT GENERATION")
print("=" * 70)

for prompt in prompts:
    start = time.time()
    output = generate(prompt, max_tokens=150, temperature=0.8, top_k=50)
    elapsed = time.time() - start
    tokens_generated = len(tokenizer.encode(output)) - len(tokenizer.encode(prompt))

    print(f"\n{'─' * 70}")
    print(f"Prompt: {prompt}")
    print(f"{'─' * 70}")
    print(output)
    print(f"\n  [{tokens_generated} tokens | {elapsed:.2f}s | {tokens_generated/elapsed:.1f} tok/s]")

In [ ]:
# Custom prompt — edit and run!
custom_prompt = "A little dog named Max loved to play in the garden"

output = generate(
    custom_prompt,
    max_tokens=200,
    temperature=0.7,    # Lower = more focused/predictable output
    top_k=40,           # Consider only top 40 candidates per step
    top_p=0.9,          # Nucleus: dynamic K based on confidence
)
print(output)

## 7. Generation Variations (Temperature Study)

Observe how temperature affects creativity vs coherence:
- **0.3**: Very safe, repetitive, stick to common patterns
- **0.6**: Balanced, natural-sounding
- **0.8**: Default — good mix of creativity and coherence
- **1.0**: Raw model distribution — more surprising choices
- **1.3**: High entropy — creative but may lose coherence

In [ ]:
prompt = "The old wizard opened his book and"

print(f"Prompt: {prompt}\n")
for temp in [0.3, 0.6, 0.8, 1.0, 1.3]:
    output = generate(prompt, max_tokens=80, temperature=temp, top_k=50)
    continuation = output[len(prompt):]
    print(f"Temperature {temp}: {continuation[:200]}")
    print()

## 8. Model Stats & Benchmarking

In [ ]:
# Throughput benchmark: measure tokens/second
# Note: this is without KV-cache, so each new token recomputes full attention
prompt = "Once upon a time in a land far away"
num_runs = 5
tokens_per_run = 100

times = []
for _ in range(num_runs):
    start = time.time()
    _ = generate(prompt, max_tokens=tokens_per_run, temperature=0.8)
    times.append(time.time() - start)

avg_time = sum(times) / len(times)
print(f"Benchmark ({num_runs} runs, {tokens_per_run} tokens each):")
print(f"  Average time: {avg_time:.2f}s")
print(f"  Throughput: {tokens_per_run / avg_time:.1f} tokens/sec")
print(f"  Model: {total_params/1e6:.1f}M params")
print(f"  Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"  GPU Memory Used: {torch.cuda.memory_allocated()/1e9:.2f} GB")